# 1 · Diagnose and evaluate

**Can a fluent answer use the wrong policy?** We will follow `day_45_opening`: a customer bought a non-defective item 45 days ago and asks for a full refund.

The starting assistant is a simulated deployed configuration with an obsolete source. No actual production deployment is implied. Read the mode, date and evidence ID below before interpreting a score.

**Presentation route:** run all cells before presenting, start at **Inspect one answer, then its source**, and return to the two short code views. This changes the reading order only; it makes no duplicate application calls.


In [ ]:
import importlib
import os, sys, json
from pathlib import Path

# Works in local Jupyter and a Databricks Git folder; notebooks have no __file__.
start = Path(os.environ.get("WORKSHOP_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "west_workshop" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set WORKSHOP_PROJECT_ROOT to the workshop checkout, then restart this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)
MODE = os.environ.get("WORKSHOP_MODE", "saved")  # Explicit opt-in before any model calls.


In [ ]:
from IPython.display import display, HTML, Code
from west_workshop.repair_workshop import notebook_session, show_case, evaluation_table

session = notebook_session(MODE)
print(json.dumps(session.banner(), indent=2))


## Code view 1: one business check

The shared release scorer below checks the declared eligibility against the reviewed label. `eligibility_declaration` also requires exactly one declaration: a correct first label followed by a contradictory second label fails. The semantic judge separately checks the explanation and transaction claims.


In [ ]:
import inspect
from west_workshop.runtime import policy_decision_value

display(Code(inspect.getsource(policy_decision_value), language="python"))


## Code view 2: the actual MLflow call

`data` contains the ten named cases. `predict_fn` is the real traced application. `scorers` contains deterministic checks and the versioned semantic judge. Results keep the same case IDs.

In live mode this cell makes one application request per case. In saved mode it reads the original result. Reopening an already recorded stage does not generate another batch. The full adapter and persistence code remain editable in `west_workshop/`.


In [ ]:
if session.needs("deployed"):
    import mlflow
    with session.evaluation("deployed") as run:
        result = mlflow.genai.evaluate(
            data=run.data, predict_fn=run.predict_fn, scorers=run.scorers,
        )
        session.record_evaluation("deployed", result, run)

deployed = session.recorded("deployed")
print("Recorded cases:", deployed["row_count"], "| Complete:", deployed["complete"])
if not deployed["complete"]:
    raise RuntimeError("This evaluation is incomplete. Its partial evidence is preserved; inspect the failure before retrying in a new directory.")


## Inspect one answer, then its source

**Pause for 15 seconds:** Does this answer deserve a full refund, store credit, or support review? You can answer without reading Python. The presenter will read the answer and current policy aloud.


In [ ]:
opening = show_case(session, "deployed", "day_45_opening")


In [ ]:
from west_workshop.data import CURRENT_POLICY

print("AUTHORITATIVE CURRENT POLICY\n" + CURRENT_POLICY)
print("\nSOURCE RETRIEVED IN THIS TRACE")
retrieval = [span for span in opening.get("spans", []) if span["name"] == "retrieve_refund_policy"]
print(json.dumps(retrieval[0]["outputs"], indent=2) if retrieval else "Retrieval span is missing from this evidence. The source cannot be verified from this recorded trace.")


At day 45, a non-defective item qualifies for store credit under the current policy. An assistant that follows the obsolete 90-day window can sound helpful while giving the wrong entitlement. The trace locates the wrong source; the score makes that failure repeatable.

A correct label still does not prove a correct explanation. For example, a day-30 answer can select `full_refund` correctly while explaining an obsolete 90-day rule.


In [ ]:
display(HTML(evaluation_table(deployed)))


## What did these checks establish?

Format and label checks establish narrow properties. The phrase tripwire catches some false claims that a transaction happened. The semantic judge examines meaning, and can itself be wrong. A failed check is evidence to inspect, not an automatic explanation of root cause.

**Consider:** which check could catch a correct label with a wrong 90-day explanation? The semantic judge should; the label check cannot.

Next: **02_compare_and_decide.ipynb**. We test the evaluator, hold the cases and checks fixed, and decide whether to accept the proposed repair under a declared workshop policy.
